## Reading files

A WhatsApp export is a plain `.txt` file where **one line is usually one message**, but not
always: a message that contains newlines is written across several lines, and only the first
one carries a timestamp. Everything below is built around that single fact.

We turn each export into a tidy DataFrame with three columns — `timestamp`, `sender`,
`message` — in **one pass over the file**:

| Stage | What happens |
| --- | --- |
| **Normalise** | Fix invisible Unicode characters, once, for the whole file |
| **Drop** | Throw away lines that carry no conversational signal |
| **Scrub** | Strip noise that sits *inside* an otherwise good message |
| **Parse** | Split each line into timestamp / sender / message, re-attaching continuation lines |
| **Timestamps** | Convert the timestamp column to real datetimes — vectorised, not row by row |


### The cleaning rules

**Dropped entirely** — the whole line goes away:

| # | Rule | Example line |
| --- | --- | --- |
| 1 | WhatsApp encryption notice | `dd/mm/yyyy, hh:mm - Messages and calls are end-to-end encrypted...` |
| 2 | Attachment placeholders | `... - Person: <Media omitted>` |
| 3 | Anything containing an email address | `... - Person: example@gmail.com` |
| 4 | Anything containing a link | `... - Person: https://www.example.com/` |
| 5 | Deleted messages | `... - Person: You deleted this message` |
| 6 | Group-creation notices | `... - Person created group "group name"` |
| 7 | "Added you" notices | `... - Person added you` |
| 8 | Messages whose entire body is `null` | `... - Person: null` |

Rules 1–7 are decided from the raw line, so they are folded into **one** regex
(`DROP_LINE`) and cost a single scan per line. Rule 8 is different: `null` is only
junk when it *is* the whole message — a line like `is it null or empty?` must survive —
so it is applied after the message body has been split out.

**Scrubbed inline** — the line stays, the noise is removed:

| # | Rule | Before | After |
| --- | --- | --- | --- |
| 9 | Edit marker | `hey, how are you? <This message was edited>` | `hey, how are you?` |
| 10 | Tagging | `@person are you coming?` | `are you coming?` |

Removing text mid-line leaves double spaces behind, so a scrub is always followed by a
whitespace collapse — otherwise those gaps end up in the training corpus.

**Normalised** — applied to the whole file before anything else, so every later rule sees
canonical text:

- `\u202F` (narrow no-break space, which iOS puts before `AM`/`PM`) and `\u00A0`
  (non-breaking space) become ordinary spaces.
- `\u200E` / `\u200F` (left-to-right and right-to-left marks) are deleted.

These are invisible in an editor but break both timestamp parsing and regex matching,
which makes them miserable to debug — worth doing first and forgetting about.


In [1]:
import re
from pathlib import Path

import pandas as pd

# Every pattern is compiled once, here, instead of on each call. These run over
# hundreds of thousands of lines, so the compile cost should be paid a single time.

# --- Normalisation -------------------------------------------------------
# str.translate() applies the whole table in one C-level pass, which is both
# faster and clearer than chaining several .replace() calls.
UNICODE_FIXES = str.maketrans({
    "\u202f": " ",   # narrow no-break space (iOS puts it before AM/PM)
    "\u00a0": " ",   # non-breaking space
    "\u200e": None,  # left-to-right mark  -> delete
    "\u200f": None,  # right-to-left mark  -> delete
})

# --- Rules 1-7: drop the whole line ---------------------------------------
# One alternation instead of eight separate checks: the line is scanned once,
# and the rules stay readable because re.VERBOSE lets us comment each branch.
# (In VERBOSE mode whitespace is ignored, so literal spaces are escaped as "\ ".)
DROP_LINE = re.compile(
    r"""
      Messages\ and\ calls\ are\ end-to-end\ encrypted  # 1 encryption notice
    | <Media\ omitted>                                    # 2 attachments
    | [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}       # 3 email address
    | https?://\S+                                        # 4 link
    | You\ deleted\ this\ message                         # 5 deleted message
    | created\ group                                      # 6 group created
    | added\ you                                          # 7 added to group
    """,
    re.VERBOSE,
)

# --- Rules 9-10: scrub inside the line ------------------------------------
INLINE_NOISE = re.compile(r"<This message was edited>|@\w+")
EXTRA_SPACE = re.compile(r"[ \t]{2,}")

# --- Message header -------------------------------------------------------
# Anchored with ^ and matched via .match(), so a continuation line is rejected
# on its very first character. Handles both export flavours:
#   Android:  14/07/2026, 18:30 - Riya: hey
#   iOS:     [15/07/2026, 9:05 AM] ~ Arjun: hey
MESSAGE_HEADER = re.compile(
    r"""^\[?                                  # iOS wraps the timestamp in [ ]
        (\d{1,2}/\d{1,2}/\d{2,4},\s           # date
         \d{1,2}:\d{2}(?::\d{2})?             # time, seconds optional
         (?:\s?[APap][Mm])?)                   # AM/PM, iOS only
        \]?
        \s?[-~]?\s?                            # " - " on Android, " ~ " on iOS
        ([^:]{1,60}):\                          # sender, up to the first colon
        (.*)$                                   # the message itself
    """,
    re.VERBOSE,
)

### Timestamps: the part that fails silently

`14/07/2026` is unambiguous — there is no 14th month. `01/03/2026` is not: it is
1 March to most of the world and 3 January in the US. WhatsApp writes whichever order
the exporting phone's locale uses and gives no hint about which one it chose.

pandas defaults to **month first**, so a day-first export is parsed as a *mix* of correct
and silently swapped dates — the unambiguous ones land correctly, the ambiguous ones
don't. Nothing raises, nothing is `NaT`, and the damage only shows up much later as
chats that appear out of order.

So we infer the order from the data instead of assuming it: scan the dates for the first
one that can only be read a single way (a component greater than 12) and let it decide.
The same scan also tells us the year width, whether seconds are present, and whether the
clock is 12- or 24-hour — enough to build an **exact** format string.

That exactness is worth real time. `format="mixed"` makes pandas re-sniff the format of
every single value; an explicit format is roughly an order of magnitude faster. We use
`"mixed"` only as a fallback for the rows the exact format failed on, which is what keeps
files that genuinely mix Android and iOS lines working.


In [2]:
# Splits a timestamp into its parts so we can work out the right format string.
STAMP_SHAPE = re.compile(
    r"^(\d{1,2})/(\d{1,2})/(\d{2,4}), (\d{1,2}):(\d{2})(:\d{2})?(\s?)([APap][Mm])?$"
)


def infer_datetime_format(stamps: pd.Series) -> tuple[str, bool]:
    """Work out the strftime format used by this export.

    Returns the format string and the day-first flag, so the caller can reuse the
    flag for the fallback parse. Returns ("", True) if no timestamp is recognisable.
    """
    # Day-first or month-first? Only a component greater than 12 can settle it.
    day_first = True  # sensible default: WhatsApp's most common layout
    for stamp in stamps:
        parts = STAMP_SHAPE.match(stamp)
        if parts is None:
            continue
        first, second = int(parts[1]), int(parts[2])
        if first > 12:      # 14/07 -> can only be day/month
            day_first = True
            break
        if second > 12:     # 07/14 -> can only be month/day
            day_first = False
            break
        # Both <= 12: ambiguous, keep looking.

    # Read the remaining details off the first well-formed timestamp. A file that
    # mixes layouts will only match the majority here; the rest are picked up by
    # the "mixed" fallback in parse_timestamps().
    shape = next((m for m in map(STAMP_SHAPE.match, stamps) if m), None)
    if shape is None:
        return "", day_first

    date = "%d/%m" if day_first else "%m/%d"
    year = "%Y" if len(shape[3]) == 4 else "%y"
    hour = "%I" if shape[8] else "%H"          # 12-hour clock only when AM/PM is present
    seconds = ":%S" if shape[6] else ""
    meridiem = f"{shape[7]}%p" if shape[8] else ""   # shape[7] preserves the space, if any
    return f"{date}/{year}, {hour}:%M{seconds}{meridiem}", day_first


def parse_timestamps(stamps: pd.Series) -> pd.Series:
    """Convert a column of timestamp strings to datetimes in one vectorised call."""
    fmt, day_first = infer_datetime_format(stamps)
    if not fmt:
        return pd.Series(pd.NaT, index=stamps.index)

    parsed = pd.to_datetime(stamps, format=fmt, errors="coerce")

    # Retry only what the exact format could not handle (e.g. iOS lines inside an
    # otherwise Android export). Usually this is zero rows and costs nothing.
    missing = parsed.isna()
    if missing.any():
        parsed[missing] = pd.to_datetime(
            stamps[missing], format="mixed", dayfirst=day_first, errors="coerce"
        )
    return parsed

### One pass over the file

The reader below walks the lines once and keeps three parallel lists. Two details are
worth pointing out:

**Continuation lines.** If a line has no timestamp header it is not a new message — it is
the rest of the previous one, so it is appended to the last body we collected. A
continuation line that arrives before any message (a stray header, say) has nothing to
attach to and is dropped.

**Columns, not rows.** Building three lists and handing them to `pd.DataFrame` at the end
means pandas allocates each column once. Appending row by row, or calling `pd.to_datetime`
per value, makes the cost scale with the number of messages instead of the number of
columns — the single biggest thing that slows a loop like this down.


In [3]:
def read_whatsapp_chat(file_path) -> pd.DataFrame:
    """Read one WhatsApp export into a DataFrame of timestamp / sender / message."""
    text = Path(file_path).read_text(encoding="utf-8").translate(UNICODE_FIXES)

    stamps: list[str] = []
    senders: list[str] = []
    bodies: list[str] = []

    for line in text.splitlines():
        if DROP_LINE.search(line):                     # rules 1-7
            continue

        # Scrub inline noise, then close the gaps it left behind (rules 9-10).
        line = EXTRA_SPACE.sub(" ", INLINE_NOISE.sub("", line)).strip()
        if not line:
            continue

        header = MESSAGE_HEADER.match(line)
        if header:
            stamp, sender, body = header.groups()
            stamps.append(stamp.strip())
            senders.append(sender.strip())
            bodies.append(body.strip())
        elif bodies:
            # No header: this is the next line of the message we are already building.
            bodies[-1] = f"{bodies[-1]}\n{line}"

    df = pd.DataFrame({"timestamp": stamps, "sender": senders, "message": bodies})

    # Rule 8, plus messages left empty by scrubbing (e.g. a body that was only a tag).
    keep = df["message"].ne("") & df["message"].str.lower().ne("null")
    df = df[keep].reset_index(drop=True)

    df["timestamp"] = parse_timestamps(df["timestamp"])
    return df

### Checking the rules actually fire

The chats in `../Data/private` only happen to contain media placeholders, links and tags.
Six of the ten rules are never exercised by them, which means a broken rule can sit there
unnoticed for a long time — so here is a small sample that triggers every rule at once,
plus a multi-line message and an iOS-format line.

Run this whenever you change a pattern. It takes milliseconds and it is the difference
between "the code ran" and "the code did what the table above says it does".


In [4]:
import tempfile

SAMPLE = """\
14/07/2026, 18:30 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
14/07/2026, 18:31 - Riya: <Media omitted>
14/07/2026, 18:32 - Riya: mail me at riya@example.com
14/07/2026, 18:33 - Riya: https://www.example.com/page
14/07/2026, 18:34 - Riya: hey, how are you? <This message was edited>
14/07/2026, 18:35 - Riya: You deleted this message
14/07/2026, 18:36 - Riya: null
14/07/2026, 18:37 - Riya created group "study group"
14/07/2026, 18:38 - Riya added you
14/07/2026, 18:39 - Riya: @arjun are you coming?
14/07/2026, 18:40 - Riya: line one
continued on the next line
[15/07/2026, 9:05 AM] ~ Arjun: iOS format works too
"""

EXPECTED = [
    ("2026-07-14 18:34", "Riya", "hey, how are you?"),
    ("2026-07-14 18:39", "Riya", "are you coming?"),
    ("2026-07-14 18:40", "Riya", "line one\ncontinued on the next line"),
    ("2026-07-15 09:05", "Arjun", "iOS format works too"),
]

sample_file = Path(tempfile.mkdtemp()) / "sample.txt"
sample_file.write_text(SAMPLE, encoding="utf-8")

actual = [
    (str(t)[:16], s, m)
    for t, s, m in read_whatsapp_chat(sample_file).itertuples(index=False)
]

assert actual == EXPECTED, "\n".join(
    ["cleaning rules did not behave as documented:"]
    + [f"  expected {row}" for row in EXPECTED]
    + [f"  actual   {row}" for row in actual]
)
print(f"all 10 rules behave as documented ({len(actual)} messages kept out of 13 lines)")

all 10 rules behave as documented (4 messages kept out of 13 lines)


### Reading every chat

`all_chats` maps each file name to its DataFrame. Sorting the paths keeps the order — and
therefore the combined text further down — stable from run to run, which matters because
the tokenizer in the next notebook is trained on exactly that text.


In [5]:
data_directory = Path("../Data/private")

all_chats = {
    file.stem: read_whatsapp_chat(file)
    for file in sorted(data_directory.glob("*.txt"))
}

for name, chat in all_chats.items():
    unparsed = chat["timestamp"].isna().sum()
    note = f"  ({unparsed} unparsed timestamps)" if unparsed else ""
    print(f"{name:<22} {len(chat):>5} messages  "
          f"{chat['timestamp'].min():%Y-%m-%d} -> {chat['timestamp'].max():%Y-%m-%d}{note}")

book_club                386 messages  2026-01-02 -> 2026-12-27
bro_chat                 927 messages  2026-03-03 -> 2026-06-21
college_project          904 messages  2026-08-06 -> 2027-02-20
cousins_group            495 messages  2026-02-18 -> 2028-01-01
cricket_gang             471 messages  2026-03-14 -> 2026-08-14
dad_and_daughter         576 messages  2026-01-03 -> 2028-01-01
exam_prep                498 messages  2025-12-03 -> 2026-07-29
family_group             917 messages  2026-01-03 -> 2026-05-24
foodie_chat              500 messages  2026-01-02 -> 2026-07-31
freelance_client         381 messages  2026-01-03 -> 2027-09-14
gym_buddies              367 messages  2026-02-03 -> 2026-10-27
highschool_couple        391 messages  2026-07-14 -> 2026-07-15
hostel_wing              471 messages  2026-01-15 -> 2026-07-31
landlord_tenant          496 messages  2026-01-02 -> 2027-02-05
long_distance            502 messages  2025-01-05 -> 2025-12-31
mom_and_son              177 messages  2

## Text sequence

The next notebook applies BPE to a single stream of text, so the messages are joined into
one string. One `" ".join(...)` over a generator builds the result in a single allocation;
growing a string with `+=` in a loop re-copies everything each time, which turns a linear
job into a quadratic one on larger corpora.


In [6]:
text_sequence = " ".join(
    message
    for chat in all_chats.values()
    for message in chat["message"]
)

len(text_sequence)

575440

Finally, save the sequence so `2_BytePairEncoding.ipynb` can pick it up.

In [7]:
out_path = Path("../output/combined_text.txt")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(text_sequence, encoding="utf-8")

print(f"wrote {out_path.resolve()} ({len(text_sequence):,} characters)")

wrote /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/TrainYourOwnLLM-Tutorial/output/combined_text.txt (575,440 characters)
